In [ ]:
!pip install pdfminer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 47.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 42.7 MB/s eta 0:00:00
  Created wheel for pdfminer: filename=pdfminer-20191125-py3-none-any.whl size=6140075 sha256=42b72e8c346321a9f5da22c0d9201f56940f6b553ad1f37da97c72f48fc60166
  Stored in directory: /root/.cache/pip/wheels/4e/c1/68/f7bd0a8f514661f76b5cbe3b5f76e0033d79f1296012cbbf72
Successfully built pdfminer


In [ ]:

!pip install pdfminer.six

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 55.0 MB/s eta 0:00:00


In [ ]:
from pdfminer.high_level import extract_text
from pdfminer.layout import LAParams
from google.colab import files
from transformers import T5ForConditionalGeneration, T5Tokenizer
import os


In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd

text_ans_data = pd.read_csv('text_ans_pairs.csv')
text_ans_data.head()

text_ans_data['input'] = 'Generate question: ' + text_ans_data['input']

text_ans_data.head()

from transformers import T5Tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")

text_ans_data['input_tokenized'] = text_ans_data['input'].apply(lambda x: tokenizer(x, max_length=512, truncation=True)['input_ids'])
text_ans_data['output_tokenized'] = text_ans_data['output'].apply(lambda x: tokenizer(x, max_length=512, truncation=True)['input_ids'])

print(text_ans_data['output_tokenized'][0])
print(text_ans_data['output_tokenized'][1000])

print(text_ans_data['input_tokenized'][0])
print(text_ans_data['input_tokenized'][1000])

text_ans_data.head()

training_data = text_ans_data[['input_tokenized','output_tokenized']]
final_json = training_data.to_json(orient='records', indent=4)

with open('final_json.json', 'w') as file:
    file.write(final_json)

training_data_2 = text_ans_data[['input','output']]
final_json_2 = training_data_2.to_json(orient='records', indent=4)

with open('final_json_2.json', 'w') as file:
    file.write(final_json_2)

In [ ]:

pip install datasets

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments
import json
from datasets import Dataset

# Load the model and tokenizer
model_name = "t5-small"  # Or any model of your choice
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)

# Parse the JSON string into a Python list of dictionaries
with open('final_json_2.json', 'r') as file:
    final_json_2 = json.load(file)

# Extract input and output columns
data_dict = {
    "input": [item["input"] for item in final_json_2],
    "output": [item["output"] for item in final_json_2]
}

# Create dataset from dictionary
train_dataset = Dataset.from_dict(data_dict)

# Tokenization function with consistent max_length and padding
def preprocess_function(examples):
    inputs = examples['input']
    targets = examples['output']

    # Tokenize inputs with padding and truncation
    model_inputs = tokenizer(inputs, max_length=145, padding="max_length", truncation=True)

    # Tokenize the targets (labels) with padding and truncation
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=145, padding="max_length", truncation=True)

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Apply the tokenization to the dataset
tokenized_dataset = train_dataset.map(preprocess_function, batched=True)

# Training setup
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=30,
    per_device_train_batch_size=8,
    remove_unused_columns=False,  # Avoid automatic column removal
    logging_dir='./logs',  # Directory for TensorBoard logs
    logging_steps=10,     # Log every 10 steps
)

# Initialize the Trainer with the tokenized dataset
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

# Train the model
trainer.train()



In [ ]:
# After training is completed, save the model and tokenizer
model_save_path = './saved_model'

# Save the trained model
trainer.save_model(model_save_path)

# Save the tokenizer as well (to ensure compatibility during loading/inference)
tokenizer.save_pretrained(model_save_path)

print(f"Model and tokenizer saved to {model_save_path}")



In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the saved model and tokenizer
model_save_path = './saved_model'
model = T5ForConditionalGeneration.from_pretrained(model_save_path)
tokenizer = T5Tokenizer.from_pretrained(model_save_path)

print("Model and tokenizer loaded successfully.")
